## Round 1 — SQL / Problem Solving

1. **Top 3 Drivers by Average Rating**
   Write an SQL query to find the top 3 drivers by average rating, including only drivers with more than 100 trips.

2. **Multiple Rides Within 10 Minutes**
   Find users who booked a ride more than once within a 10-minute window using a self-join or window functions.

3. **Most Frequent Route**
   Given a `rides` table, find the most frequently traveled route in the past 30 days.

4. **Duplicate Detection**
   Write an efficient query to detect duplicates based on `customer_id` and timestamp, but return only the latest record.

5. **Week-over-Week Trip Count Difference**
   Find the difference in trip counts between two consecutive weeks for each driver.

---


#### 1. Top 3 Drivers by Average Rating

Assume the table is:

`rides(driver_id, ride_id, rating, ride_date)`

### SQL Query

```sql
SELECT
    driver_id,
    COUNT(ride_id) AS total_trips,
    AVG(rating) AS avg_rating
FROM rides
GROUP BY driver_id
HAVING COUNT(ride_id) > 100
ORDER BY avg_rating DESC
LIMIT 3;
```

### Interview Explanation — Short

1. `GROUP BY driver_id` → group rides by driver.
2. `COUNT(ride_id)` → calculate total trips.
3. `AVG(rating)` → calculate average rating.
4. `HAVING COUNT(ride_id) > 100` → keep only drivers with more than 100 trips.
5. `ORDER BY avg_rating DESC` → highest-rated drivers first.
6. `LIMIT 3` → return the top 3.

**Key point:** Use `HAVING`, not `WHERE`, because the filter is based on an aggregate (`COUNT`).


#### 2. Multiple Rides Within 10 Minutes

Assume the table is:

`rides(user_id, ride_id, booking_time)`

### SQL using Window Function

```sql
WITH ride_times AS (
    SELECT
        user_id,
        ride_id,
        booking_time,
        LAG(booking_time) OVER (
            PARTITION BY user_id
            ORDER BY booking_time
        ) AS previous_booking
    FROM rides
)
SELECT DISTINCT user_id
FROM ride_times
WHERE booking_time <= previous_booking + INTERVAL '10 minutes';
```

### Interview Explanation — Short

1. `PARTITION BY user_id` → analyze rides separately for each user.
2. `LAG()` → get the previous booking time.
3. Compare current booking with the previous booking.
4. If the difference is **≤ 10 minutes**, the user booked multiple rides within 10 minutes.
5. `DISTINCT` → return each user only once.

### Alternative: Self-Join

```sql
SELECT DISTINCT r1.user_id
FROM rides r1
JOIN rides r2
    ON r1.user_id = r2.user_id
   AND r1.ride_id <> r2.ride_id
   AND r2.booking_time > r1.booking_time
   AND r2.booking_time <= r1.booking_time + INTERVAL '10 minutes';
```

**Interview tip:** `LAG()` is generally cleaner and more efficient than a self-join for this type of time-window comparison.


#### 3. Most Frequent Route

Assume the table is:

`rides(ride_id, pickup_location, drop_location, ride_date)`

### SQL Query

```sql
SELECT
    pickup_location,
    drop_location,
    COUNT(*) AS trip_count
FROM rides
WHERE ride_date >= CURRENT_DATE - INTERVAL '30 days'
GROUP BY
    pickup_location,
    drop_location
ORDER BY trip_count DESC
LIMIT 1;
```

### Interview Explanation — Short

1. Filter rides from the **last 30 days** using `WHERE`.
2. Group by `pickup_location` and `drop_location` to identify each route.
3. `COUNT(*)` → calculate the number of trips for each route.
4. Sort by trip count in descending order.
5. `LIMIT 1` → return the **most frequently traveled route**.

### If multiple routes can tie for #1

Use `DENSE_RANK()`:

```sql
WITH route_counts AS (
    SELECT
        pickup_location,
        drop_location,
        COUNT(*) AS trip_count
    FROM rides
    WHERE ride_date >= CURRENT_DATE - INTERVAL '30 days'
    GROUP BY pickup_location, drop_location
)
SELECT *
FROM (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY trip_count DESC) AS rnk
    FROM route_counts
) t
WHERE rnk = 1;
```

**Interview tip:** Use `DENSE_RANK()` when the requirement is to return **all routes tied for the highest frequency**.


#### 4. Duplicate Detection

Assume the table is:

`customer_data(customer_id, event_timestamp, record_id, ...)`

### SQL Query — Using `ROW_NUMBER()`

```sql
WITH ranked_data AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id, event_timestamp
            ORDER BY record_id DESC
        ) AS rn
    FROM customer_data
)
SELECT *
FROM ranked_data
WHERE rn = 1;
```

### Interview Explanation — Short

1. `PARTITION BY customer_id, event_timestamp` → identifies duplicate records.
2. `ORDER BY record_id DESC` → keeps the latest/highest record.
3. `ROW_NUMBER()` assigns `1` to the latest record.
4. `WHERE rn = 1` → returns only the latest record for each duplicate group.

### If timestamp itself determines the latest record

If duplicates are based on `customer_id` but you want the **latest timestamp**:

```sql
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM customer_data
) t
WHERE rn = 1;
```






#### 5. Week-over-Week Trip Count Difference



### Assume `trips` table

| trip_id | driver_id | trip_date  |
| ------: | --------: | ---------- |
|       1 |       101 | 2026-09-01 |
|       2 |       101 | 2026-09-02 |
|       3 |       101 | 2026-09-03 |
|       4 |       101 | 2026-09-09 |
|       5 |       101 | 2026-09-10 |
|       6 |       102 | 2026-09-01 |
|       7 |       102 | 2026-09-02 |
|       8 |       102 | 2026-09-08 |
|       9 |       102 | 2026-09-09 |
|      10 |       102 | 2026-09-10 |

### SQL Query

```sql
WITH weekly_trips AS (
    SELECT
        driver_id,
        DATE_TRUNC('week', trip_date) AS week,
        COUNT(*) AS trip_count
    FROM trips
    GROUP BY driver_id, DATE_TRUNC('week', trip_date)
)
SELECT
    driver_id,
    week,
    trip_count,
    LAG(trip_count) OVER (
        PARTITION BY driver_id
        ORDER BY week
    ) AS previous_week_count,
    trip_count - LAG(trip_count) OVER (
        PARTITION BY driver_id
        ORDER BY week
    ) AS week_over_week_difference
FROM weekly_trips
ORDER BY driver_id, week;
```

### Expected Output

| driver_id | week       | trip_count | previous_week_count | week_over_week_difference |
| --------: | ---------- | ---------: | ------------------: | ------------------------: |
|       101 | 2026-08-31 |          3 |                NULL |                      NULL |
|       101 | 2026-09-07 |          2 |                   3 |                    **-1** |
|       102 | 2026-08-31 |          2 |                NULL |                      NULL |
|       102 | 2026-09-07 |          3 |                   2 |                    **+1** |

### Interview Explanation

**`LAG()`** gets the previous week's trip count, and then:

**Current Week Trips − Previous Week Trips = WoW Difference**

So, for driver **101**: `2 - 3 = -1` → **1 fewer trip**.



## Round 2 — DSA / Algorithm

1. **Running Median**
   Given a stream of integers, design a data structure to return the median at any time.

2. **Cycle Detection in Directed Graph**
   Detect a cycle in a directed graph representing trip routes between cities.

3. **Merge K Sorted Linked Lists**
   Merge K sorted linked lists efficiently.

4. **LRU Cache**
   Implement an LRU (Least Recently Used) cache.

5. **Longest Continuous Ride Streak**
   Given a list of ride timestamps, find the longest continuous ride streak.

---


#### 1. Running Median


**Problem:** Given a stream of integers, return the median after each insertion.

### Approach: Two Heaps

Use **two heaps**:

* **Max Heap (`lower`)** → stores smaller half of numbers.
* **Min Heap (`upper`)** → stores larger half of numbers.
* Keep their sizes balanced, with `lower` having at most one extra element.

### Python Solution

```python
import heapq

class MedianFinder:

    def __init__(self):
        self.lower = []  # max heap (use negative values)
        self.upper = []  # min heap

    def add_number(self, num):
        # Add to max heap
        heapq.heappush(self.lower, -num)

        # Ensure every lower element <= upper elements
        if self.lower and self.upper and (-self.lower[0] > self.upper[0]):
            val = -heapq.heappop(self.lower)
            heapq.heappush(self.upper, val)

        # Balance heap sizes
        if len(self.lower) > len(self.upper) + 1:
            val = -heapq.heappop(self.lower)
            heapq.heappush(self.upper, val)

        elif len(self.upper) > len(self.lower):
            val = heapq.heappop(self.upper)
            heapq.heappush(self.lower, -val)

    def get_median(self):
        if len(self.lower) > len(self.upper):
            return -self.lower[0]

        return (-self.lower[0] + self.upper[0]) / 2
```

### Example

Stream:

```text
5 → 2 → 10 → 4 → 8
```

| Insert | Numbers        | Median |
| -----: | -------------- | -----: |
|      5 | `[5]`          |      5 |
|      2 | `[2,5]`        |    3.5 |
|     10 | `[2,5,10]`     |      5 |
|      4 | `[2,4,5,10]`   |    4.5 |
|      8 | `[2,4,5,8,10]` |      5 |

### Complexity

* **Insertion:** `O(log n)`
* **Get median:** `O(1)`
* **Space:** `O(n)`

**Interview key point:** Two heaps avoid sorting the entire stream every time, which would cost **O(n log n)** per median calculation.


#### 2. Cycle Detection in Directed Graph


**Problem:** Given a directed graph representing trip routes between cities, detect whether there is a cycle.

### Approach: DFS + Recursion Stack

Use two arrays/sets:

* `visited` → city has already been explored.
* `rec_stack` → city is currently in the DFS path.
* If we reach a city already in `rec_stack`, **a cycle exists**.

### Python Solution

```python id="d9x4pk"
from collections import defaultdict

def has_cycle(graph):
    visited = set()
    rec_stack = set()

    def dfs(city):
        if city in rec_stack:
            return True

        if city in visited:
            return False

        visited.add(city)
        rec_stack.add(city)

        for next_city in graph[city]:
            if dfs(next_city):
                return True

        rec_stack.remove(city)
        return False

    for city in graph:
        if dfs(city):
            return True

    return False
```

### Assume Trip Routes

```python id="8z2jtc"
graph = {
    "Delhi": ["Mumbai"],
    "Mumbai": ["Bangalore"],
    "Bangalore": ["Chennai"],
    "Chennai": ["Delhi"]
}
```

Graph:

```text
Delhi → Mumbai → Bangalore → Chennai
  ↑                         ↓
  └─────────────────────────┘
```

**Output:**

```text
Cycle exists: True
```

### Complexity

* **Time:** `O(V + E)`
* **Space:** `O(V)`

**Interview key point:** In a directed graph, finding an edge to a node currently present in the **recursion stack** confirms a cycle.


#### 3. Merge K Sorted Linked Lists


**Problem:** Given `K` sorted linked lists, merge them into one sorted linked list efficiently.

### Approach: Min Heap

Use a **min heap** to always select the smallest current node among the `K` lists.

### Python Solution

```python id="m5k2qa"
import heapq

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def merge_k_lists(lists):
    min_heap = []

    # Add first node of each list
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(min_heap, (node.val, i, node))

    dummy = ListNode(0)
    current = dummy

    while min_heap:
        val, i, node = heapq.heappop(min_heap)

        current.next = node
        current = current.next

        # Add next node from same list
        if node.next:
            heapq.heappush(
                min_heap,
                (node.next.val, i, node.next)
            )

    return dummy.next
```

### Example

**Input:**

```text
List 1: 1 → 4 → 7
List 2: 2 → 5 → 8
List 3: 3 → 6 → 9
```

**Output:**

```text
1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9
```

### Why Min Heap?

At any point, the heap contains at most **K nodes** — one candidate from each list.

```text
Heap
 ↓
[1, 2, 3]

Pop 1 → add next 4
Heap → [2, 3, 4]

Pop 2 → add next 5
Heap → [3, 4, 5]
```

### Complexity

* **N** = total number of nodes
* **K** = number of linked lists
* **Time:** `O(N log K)`
* **Space:** `O(K)`

**Interview key point:** A min heap is more efficient than repeatedly scanning all `K` lists, which would take `O(N × K)`.


#### 4. LRU Cache

**Problem:** Implement an LRU Cache that supports `get()` and `put()` in **O(1)** time.

### Approach

Use **HashMap + Doubly Linked List**:

* **HashMap** → gives O(1) lookup.
* **Doubly Linked List** → maintains usage order.
* **Most recently used** → near the head.
* **Least recently used** → near the tail.
* When cache is full, remove the node from the tail.

### Python Solution

```python id="lru82"
class Node:
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None


class LRUCache:

    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}

        # Dummy head and tail
        self.head = Node()
        self.tail = Node()

        self.head.next = self.tail
        self.tail.prev = self.head

    def remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def add_to_front(self, node):
        node.next = self.head.next
        node.prev = self.head

        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.cache:
            return -1

        node = self.cache[key]

        # Mark as recently used
        self.remove(node)
        self.add_to_front(node)

        return node.value

    def put(self, key, value):
        if key in self.cache:
            node = self.cache[key]
            node.value = value

            self.remove(node)
            self.add_to_front(node)
            return

        node = Node(key, value)
        self.cache[key] = node
        self.add_to_front(node)

        # Remove least recently used
        if len(self.cache) > self.capacity:
            lru = self.tail.prev
            self.remove(lru)
            del self.cache[lru.key]
```

### Example

Capacity = `2`

```text
put(1, "A")
put(2, "B")

Cache:
1 → 2

get(1)
```

`1` becomes recently used:

```text
1 → 2
```

Now:

```text
put(3, "C")
```

`2` is least recently used, so it gets removed:

```text
3 → 1
```

Therefore:

```text
get(2) → -1
get(3) → "C"
get(1) → "A"
```

### Complexity

| Operation |      Complexity |
| --------- | --------------: |
| `get()`   |        **O(1)** |
| `put()`   |        **O(1)** |
| Space     | **O(capacity)** |

**Interview key point:** The **HashMap provides O(1) lookup**, while the **Doubly Linked List provides O(1) insertion, deletion, and ordering**.


#### 5. Longest Continuous Ride Streak



**Problem:** Given a list of ride timestamps, find the longest streak of **consecutive rides**, where each ride occurs on the next consecutive day.

### Example

```text
Ride dates:
2026-09-01
2026-09-02
2026-09-03
2026-09-05
2026-09-06
2026-09-10
```

Longest continuous streak:

```text
2026-09-01 → 2026-09-02 → 2026-09-03
```

**Answer = 3 days**

### Python Solution

```python
from datetime import datetime

def longest_ride_streak(timestamps):

    dates = sorted(set(
        datetime.strptime(ts, "%Y-%m-%d").date()
        for ts in timestamps
    ))

    if not dates:
        return 0

    longest = 1
    current = 1

    for i in range(1, len(dates)):

        if (dates[i] - dates[i - 1]).days == 1:
            current += 1
        else:
            current = 1

        longest = max(longest, current)

    return longest
```

### Test

```python
rides = [
    "2026-09-01",
    "2026-09-02",
    "2026-09-03",
    "2026-09-05",
    "2026-09-06",
    "2026-09-10"
]

print(longest_ride_streak(rides))
```

**Output:**

```text
3
```

### Complexity

* **Sorting:** `O(N log N)`
* **Scanning:** `O(N)`
* **Overall:** **O(N log N)**
* **Space:** `O(N)`

**Interview key point:** Sort the timestamps first, then maintain a running streak whenever the difference between consecutive dates is exactly **1 day**.



## Round 3 — System Design / Data Engineering

1. **Real-Time Ride Anomaly Detection**
   Design a distributed pipeline to detect ride anomalies in real time using stream processing.

2. **Scale 100M GPS Events/Day**
   How would you scale a pipeline that ingests **100M GPS events per day** and still serves analytics queries under **1 second**?

3. **Versioned Schema System**
   Design a versioned schema system for your data warehouse to handle backward-incompatible changes.

4. **Exactly-Once Delivery**
   How would you implement exactly-once delivery in a **Kafka + Spark Structured Streaming** setup?

5. **Late-Arriving Data**
   Explain how you handle late-arriving data in real-time processing using **watermarks and state cleanup**.

---

#### 1. Real-Time Ride Anomaly Detection

## Real-Time Ride Anomaly Detection — Distributed Design

**Goal:** Detect abnormal rides in real time, such as **unusually high fare, extreme speed, GPS jumps, repeated cancellations, or suspicious ride frequency**.

### Architecture

```text
Ride Events
    ↓
Kafka
    ↓
Spark Structured Streaming
    ↓
Data Validation + Enrichment
    ↓
Windowed Aggregations
    ↓
Anomaly Detection Rules / ML Model
    ↓
 ┌───────────────┬────────────────┐
 ↓               ↓                ↓
Alert System   Redis/DB       Data Lake
(Kafka/SNS)    Dashboard       Bronze/Silver
```

### Main Steps

1. **Ingest**

   * Ride app publishes events to **Kafka**.
   * Partition by `driver_id` or `ride_id` for scalability.

2. **Stream Processing**

   * Use **Spark Structured Streaming** to consume Kafka events.
   * Parse JSON and validate schema.

3. **Enrichment**

   * Join with driver/customer metadata.
   * Enrich with location, historical driver statistics, etc.

4. **Windowed Analysis**

   * Use **5-minute / 15-minute sliding windows**.
   * Calculate metrics such as:

     * Average speed
     * Average fare
     * Number of rides per driver
     * Cancellation rate
     * GPS distance anomalies

5. **Anomaly Detection**

   * Rule-based checks for known patterns.
   * Example:

```text
speed > 150 km/h
OR
fare > 3 × driver's historical average
OR
10+ rides within 5 minutes
→ Anomaly
```

* For advanced detection, use an ML model such as **Isolation Forest**.

6. **Alerting**

   * Send detected anomalies to Kafka/SNS/Event Hub.
   * Trigger real-time alerts for operations or fraud teams.

7. **Storage**

   * Store raw events in **Bronze**.
   * Store cleaned/enriched events in **Silver**.
   * Store anomaly results and aggregates in **Gold**.

8. **Reliability**

   * Enable checkpointing and exactly-once/strong processing guarantees where supported.
   * Handle late events using **watermarks**.
   * Use Kafka offsets for recovery.
   * Configure dead-letter handling for malformed events.

### Example Event

```json
{
  "ride_id": "R101",
  "driver_id": "D25",
  "speed": 185,
  "fare": 2500,
  "timestamp": "2026-09-14T10:15:00"
}
```

**Result:**

```text
Anomaly = TRUE
Reason = "Abnormally high speed"
Severity = HIGH
```

### Interview Summary

**Kafka → Spark Structured Streaming → Validation/Enrichment → Window Aggregation → Rule/ML Detection → Alert + Delta Lake**

**Key points:** low latency, partitioning for scalability, watermarking for late data, checkpointing for fault tolerance, and monitoring for throughput/latency/backpressure.


#### 2. Scale 100M GPS Events/Day


## Scale 100M GPS Events/Day — System Design

**Goal:** Ingest **100M GPS events/day** while keeping analytics queries **<1 second**.

### Architecture

```text
GPS Devices
     ↓
Kafka / Event Hubs
     ↓
Spark Structured Streaming
     ↓
Delta Lake / ADLS
     ↓
┌──────────────────────────────┐
│ Bronze → Silver → Gold       │
└──────────────────────────────┘
     ↓
Serving Layer
     ↓
Databricks SQL / Warehouse
     ↓
BI Dashboard / API
```

### Main Design Points

1. **Ingestion**

   * Use **Kafka/Azure Event Hubs** for distributed ingestion.
   * Partition by `device_id` or `vehicle_id`.
   * Scale partitions based on throughput.

2. **Stream Processing**

   * Use **Spark Structured Streaming**.
   * Process events incrementally rather than full reloads.
   * Use checkpointing for fault tolerance.

3. **Storage**

   * Store raw data in **ADLS/Delta Lake**.
   * Use **Bronze → Silver → Gold** architecture.
   * Store GPS events in optimized **Delta/Parquet** format.

4. **Partitioning**

   * Partition primarily by **event_date/hour**.
   * Avoid excessive partitioning by `device_id`.
   * Use **Z-ORDER / clustering** on frequently filtered columns such as `device_id`, `vehicle_id`, and timestamp.

5. **Query Optimization**

   * Create **Gold-level aggregated tables** for analytics.
   * Example:

```text
GPS Events
   ↓
Hourly Vehicle Statistics
   ↓
Daily/Regional Aggregates
```

* Queries should hit aggregated Gold tables instead of scanning 100M raw events.

6. **Caching / Serving**

   * Use **Databricks SQL Warehouse** with appropriate sizing/autoscaling.
   * Cache frequently accessed datasets/results where beneficial.
   * For extremely low-latency API lookups, use a serving store such as **Redis**.

7. **Performance**

   * Enable Delta optimization/compaction.
   * Avoid small files.
   * Use predicate pushdown and column pruning.
   * Select only required columns.

8. **Scalability & Reliability**

   * Autoscale Spark workers.
   * Monitor Kafka/Event Hubs lag and Spark processing latency.
   * Use retries, checkpointing, and dead-letter handling.
   * Design for horizontal scaling rather than a single large machine.

### Interview Summary

> **Event Hubs/Kafka → Spark Streaming → Delta Bronze/Silver → Gold Aggregates → Databricks SQL/Redis**

**Key idea:** Don't try to achieve sub-second queries by scanning the **100M raw GPS events**. Pre-aggregate the data, optimize the serving tables, partition by time, cluster frequently queried dimensions, and use a dedicated low-latency serving layer when required.


#### 3. Versioned Schema System

## Versioned Schema System — Data Warehouse Design

**Goal:** Handle **backward-incompatible schema changes** without breaking existing pipelines, dashboards, or consumers.

### Architecture

```text
Source
  ↓
Schema Registry
  ↓
Validation
  ↓
Bronze
  ↓
Silver
  ↓
┌───────────────────────┐
│ Schema V1              │
│ Schema V2              │
│ Schema V3              │
└───────────────────────┘
  ↓
Gold / Data Warehouse
  ↓
Consumers / BI
```

### Main Design

1. **Schema Registry**

   * Store every schema with a unique version.
   * Example:

```text
customer_schema_v1
customer_schema_v2
customer_schema_v3
```

2. **Compatibility Rules**

   * Define backward/forward compatibility.
   * **Breaking changes** create a new major version.

Example:

```text
V1: customer_id, name, age

V2: customer_id, full_name, age
```

Renaming `name` → `full_name` is potentially breaking, so create **V2** instead of silently modifying V1.

3. **Versioned Tables**

```text
customer_v1
customer_v2
```

Existing applications continue using `customer_v1`, while new consumers migrate to `customer_v2`.

4. **Schema Metadata**

Maintain a metadata table:

| Schema   | Version | Status     | Effective Date |
| -------- | ------: | ---------- | -------------- |
| customer |      V1 | Deprecated | 2026-01-01     |
| customer |      V2 | Active     | 2026-06-01     |
| customer |      V3 | Draft      | 2026-09-01     |

5. **Migration Layer**

   * Create views or transformation logic between versions.
   * Example:

```sql
CREATE VIEW customer_v1_compat AS
SELECT
    customer_id,
    full_name AS name,
    age
FROM customer_v2;
```

This allows old consumers to continue working during migration.

6. **Deployment Process**

```text
Create Schema
      ↓
Validate Compatibility
      ↓
Register New Version
      ↓
Test Pipelines
      ↓
Deploy
      ↓
Migrate Consumers
      ↓
Deprecate Old Version
```

7. **Backward-Incompatible Changes**

For changes such as:

* Renaming columns
* Changing data types
* Removing columns
* Changing business meaning

**Never modify the existing contract directly.** Create a new schema version.

8. **Governance**

Track:

* Schema owner
* Version
* Change history
* Consumers/dependencies
* Deprecation date
* Data lineage

### Interview Summary

> **Schema Registry + Versioned Contracts + Compatibility Validation + Versioned Tables/Views + Controlled Migration**

**Key principle:** **Never break an existing schema contract silently.** Introduce a new version, migrate consumers gradually, and deprecate the old version only after all dependencies have moved.


#### 4. Exactly-Once Delivery


## Exactly-Once Delivery — Kafka + Spark Structured Streaming

**Goal:** Ensure each Kafka event is processed and written to the target **without duplicates**, even after failures/restarts.

### Architecture

```text
Kafka
  ↓
Spark Structured Streaming
  ↓
Transform / Deduplicate
  ↓
Delta Lake / Transactional Sink
```

### Main Points

1. **Kafka**

   * Use a stable `event_id` for every event.
   * Kafka provides durable storage and tracks **topic + partition + offset**.
   * Enable producer idempotence:

```properties
enable.idempotence=true
```

2. **Spark Checkpointing**

   * Configure a durable checkpoint location.

```python
query = (
    df.writeStream
      .format("delta")
      .option("checkpointLocation", "/checkpoints/rides")
      .outputMode("append")
      .start("/delta/rides")
)
```

* Spark stores offsets and streaming state in the checkpoint.
* After restart, Spark resumes from the appropriate offsets.

3. **Deduplication**

Use a unique `event_id` to protect against duplicate events:

```python
deduped = (
    df.withWatermark("event_time", "10 minutes")
      .dropDuplicates(["event_id"])
)
```

4. **Transactional Sink**

Use **Delta Lake** or another transactional sink that supports atomic commits.

```text
Process batch
     ↓
Write transaction
     ↓
Commit successfully
     ↓
Advance checkpoint
```

If the job fails before the transaction completes, the data isn't partially committed.

5. **Failure Recovery**

```text
Kafka Offset 100
     ↓
Spark processes 100
     ↓
Job crashes
     ↓
Restart from checkpoint
     ↓
Reprocess safely
     ↓
Deduplication + transactional commit
```

### Important Distinction

**Exactly-once processing ≠ exactly-once business events.**

For true business-level exactly-once behavior, combine:

> **Kafka offsets + Spark checkpointing + deterministic processing + event IDs/deduplication + transactional/idempotent sink**

### Interview Summary

> **Kafka → Spark Structured Streaming → Checkpointing → Deduplication → Transactional Delta Sink**

**Key point:** Don't rely on Kafka alone. Exactly-once behavior is an **end-to-end property**, so the source, Spark state/checkpoint, processing logic, and destination must all be designed together.


#### 5. Late-Arriving Data

## Late-Arriving Data — Watermarks & State Cleanup

**Problem:** In real-time systems, events may arrive late because of network delays, device issues, or retries.

### Example

Suppose we process rides in **5-minute windows**:

```text
Event Time:
10:01 → arrives at 10:02
10:03 → arrives at 10:04
10:04 → arrives at 10:15  ← Late event
```

### Solution: Watermarking

Use a **watermark** to define how late an event is allowed to arrive.

```python id="l8w2qp"
from pyspark.sql.functions import *

result = (
    rides
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window("event_time", "5 minutes"),
        "driver_id"
    )
    .count()
)
```

### How It Works

Assume:

```text
Watermark = Maximum observed event time - 10 minutes
```

If the maximum event time is:

```text
10:20
```

Watermark becomes:

```text
10:10
```

Events belonging to windows older than the watermark are considered **too late** and are no longer used for stateful aggregation.

### State Cleanup

Without watermarks:

```text
Incoming Events
      ↓
State Store
      ↓
State keeps growing ❌
```

With watermarks:

```text
Incoming Events
      ↓
State Store
      ↓
Watermark
      ↓
Remove old state
      ↓
Controlled memory usage ✅
```

### What Happens to Very Late Events?

For events arriving **within the watermark threshold**:

```text
→ Process normally
→ Update the corresponding window/state
```

For events arriving **after the watermark**:

```text
→ Consider too late
→ Drop or route to a late-events/DLQ table
```

### Interview Example

```text
Window: 10:00–10:05
Allowed lateness: 10 minutes

Event at 10:04 → arrives 10:08
→ Process ✅

Event at 10:04 → arrives 10:20
→ Too late → handle separately ❌
```

### Interview Summary

> **Watermark = controls how long Spark waits for late events. State cleanup = removes state for windows older than the watermark.**

**Key points:** choose the watermark based on expected business lateness, use event time rather than processing time, monitor dropped late events, and send excessively late records to a separate table/DLQ when they need reconciliation.


